# MOXEC — Cross-Dataset Aggregation

Run this **after** you've run `MOXEC_single_dataset_experiment.ipynb` on some or all of the
12 datasets in the portfolio. Each of those runs writes a `summary.json` into its own
`moxec_results/<dataset_name>/` folder; this notebook collects all of them and answers the
question the single-dataset notebook can't answer on its own: **does NSGA-II actually beat
Random Search, on average, across the portfolio — or was the near-tie on one dataset the
general case?**

Safe to re-run at any point — it only reads what's already on disk and picks up however many
dataset folders currently have a `summary.json`. You don't need all 12 finished to get a
useful (if lower-powered) preview.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

RESULTS_ROOT = Path("./moxec_results")   # match CONFIG["output_dir"] in the single-dataset notebook
# On Colab with Drive persistence, point this at your Drive folder instead, e.g.:
# RESULTS_ROOT = Path("/content/drive/MyDrive/MOXEC")

print(f"Looking for completed runs under: {RESULTS_ROOT.resolve()}")


## 1. Collect every completed dataset's summary.json

In [ ]:
summary_paths = sorted(RESULTS_ROOT.glob("*/summary.json"))
print(f"Found {len(summary_paths)} completed dataset run(s):")
for p in summary_paths:
    print(f"  {p.parent.name}")

if len(summary_paths) == 0:
    raise FileNotFoundError(
        f"No summary.json files found under {RESULTS_ROOT.resolve()}. Run the single-dataset "
        f"notebook on at least one dataset first, or fix RESULTS_ROOT above if your results "
        f"live somewhere else (e.g. Google Drive)."
    )

runs = {}
for p in summary_paths:
    with open(p) as f:
        runs[p.parent.name] = json.load(f)


## 2. Per-dataset, seed-averaged hypervolume

For each dataset, average across its seeds first — this gives ONE (MOXEC, Random, TPE) triple
per dataset, which is what a fair cross-dataset comparison needs. Averaging across seeds
*within* a dataset is legitimate (they're repeats of the same problem); pooling raw per-seed
numbers *across* datasets would not be, since datasets differ in difficulty and objective
scale.

In [ ]:
rows = []
for name, run in runs.items():
    hv_records = run.get("hypervolume_by_seed", [])
    if not hv_records:
        print(f"WARNING: {name} has no hypervolume_by_seed data -- skipping.")
        continue
    hv_df = pd.DataFrame(hv_records)
    rows.append({
        "dataset": name,
        "n_instances": run.get("dataset_meta", {}).get("n_instances"),
        "n_features": run.get("dataset_meta", {}).get("n_features"),
        "n_seeds": len(hv_df),
        "moxec_hv": hv_df["MOXEC (NSGA-II)"].mean(),
        "random_hv": hv_df["Random Search (3-obj)"].mean(),
        "tpe_hv": hv_df["TPE (MCC-only)"].mean(),
    })

portfolio_df = pd.DataFrame(rows)
portfolio_df["moxec_vs_random_gap"] = portfolio_df["moxec_hv"] - portfolio_df["random_hv"]
portfolio_df["moxec_vs_random_pct"] = 100 * portfolio_df["moxec_vs_random_gap"] / portfolio_df["random_hv"]
portfolio_df["moxec_wins"] = portfolio_df["moxec_vs_random_gap"] > 0
portfolio_df["moxec_vs_tpe_pct"] = 100 * (portfolio_df["moxec_hv"] - portfolio_df["tpe_hv"]) / portfolio_df["tpe_hv"]

portfolio_df = portfolio_df.sort_values("moxec_vs_random_pct", ascending=False).reset_index(drop=True)
portfolio_df


## 3. Does NSGA-II beat Random Search across the portfolio? (the actual question)

In [ ]:
n_datasets = len(portfolio_df)
n_wins = int(portfolio_df["moxec_wins"].sum())
mean_gap_pct = portfolio_df["moxec_vs_random_pct"].mean()

print(f"NSGA-II beat Random Search (3-obj) on {n_wins}/{n_datasets} datasets.")
print(f"Mean HV gap (MOXEC - Random) as % of Random's HV: {mean_gap_pct:+.2f}%")
print(f"Median: {portfolio_df['moxec_vs_random_pct'].median():+.2f}%")
print(f"Mean MOXEC advantage over single-objective TPE: {portfolio_df['moxec_vs_tpe_pct'].mean():+.2f}%")

if n_datasets < 8:
    print(f"\nNOTE: only {n_datasets} dataset(s) so far -- Wilcoxon below is a preview, not a "
          f"result to report. Demsar's framework wants 15-30+ datasets for real power; you "
          f"have 12 in the portfolio total, which is already on the low end for Friedman/Nemenyi. "
          f"Treat anything under ~8-10 datasets as directional only.")

if n_datasets >= 5:
    stat, p = wilcoxon(portfolio_df["moxec_hv"], portfolio_df["random_hv"])
    print(f"\nWilcoxon signed-rank (MOXEC vs. Random, paired by dataset): W={stat:.3f}, p={p:.4f}")
    if p < 0.05:
        direction = "MOXEC significantly beats" if mean_gap_pct > 0 else "Random Search significantly beats"
        print(f"-> {direction} the other, at alpha=0.05.")
    else:
        print("-> No significant difference at alpha=0.05. If this holds once all 12 datasets "
              "are in, the honest framing for the paper is: multi-objective FORMULATION is what "
              "matters (both MOXEC and Random Search crush single-objective TPE), not the choice "
              "of NSGA-II specifically as the search algorithm. That is still a real, publishable "
              "finding -- it just changes which sentence in the abstract carries the claim.")
else:
    print("\nNeed at least 5 datasets for a Wilcoxon test to mean anything -- run more datasets "
          "and re-execute this notebook.")


## 4. Per-dataset breakdown

In [ ]:
display_cols = ["dataset", "n_instances", "n_features", "moxec_hv", "random_hv",
                "moxec_vs_random_pct", "moxec_wins", "moxec_vs_tpe_pct"]
portfolio_df[display_cols].round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(portfolio_df))))
colors = ["#2a6f4f" if w else "#a33" for w in portfolio_df["moxec_wins"]]
ax.barh(portfolio_df["dataset"], portfolio_df["moxec_vs_random_pct"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("MOXEC (NSGA-II) hypervolume advantage over Random Search (%)")
ax.set_title("Per-dataset NSGA-II vs. Random Search gap")
plt.tight_layout()
plt.savefig(RESULTS_ROOT / "nsga2_vs_random_gap.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {(RESULTS_ROOT / 'nsga2_vs_random_gap.png').resolve()}")


## 5. Save the aggregated table

In [ ]:
portfolio_df.to_csv(RESULTS_ROOT / "portfolio_hv_comparison.csv", index=False)
print(f"Saved: {(RESULTS_ROOT / 'portfolio_hv_comparison.csv').resolve()}")
print(f"\nRe-run this notebook after each new dataset finishes -- it always reflects "
      f"whatever is currently on disk under {RESULTS_ROOT.resolve()}.")


---

## What this does and doesn't tell you yet

This is the mechanism for tracking the NSGA-II-vs-Random-Search question, not a final
verdict — that only comes once enough datasets are in. What to do with the answer once you
have it:

- **NSGA-II wins clearly and consistently:** keep the current abstract framing ("our
  multi-objective search algorithm outperforms baselines").
- **It's a near-tie on average (what seed 1 of Diabetic Retinopathy suggested):** reframe
  around the objective *formulation* — both MOXEC and Random Search beat single-objective
  TPE by a wide margin, which is real and defensible; NSGA-II specifically being the
  right search strategy is a separate, weaker claim that this data may not support. Say
  so directly in the paper rather than overselling the sampler.
- **Random Search wins on average:** report it. That's still an interesting, honest result
  about this problem's search landscape, and burying it is the kind of thing a careful
  reviewer catches anyway.
